# Conditional-Proposition Phase Diagram (Read-Only Analysis)

**Theoretical proposition** (derived in log space): let r = the vector of log residual ratios and
c = the centered vector of log correction factors; then multiplicative correction + renormalization reduces error if and only if
2⟨r,c⟩ > ‖c‖², i.e., **α ≡ ρ(r,c) > σ_c/(2σ_r)** (theoretical boundary α\* = σ_ratio/2).

This notebook is read-only: it displays artifacts persisted in `results/exp_r26/` and performs no computational modification.

## Overview by Layer

| Layer | Setup | Assertion discipline |
|---|---|---|
| **Synthetic Layer A** | Agent level, log-normal residuals, no aggregation (the theorem's strict domain) | Quantitative assertion: the `geo_log` / `arith_log` boundary deviates from the theoretical line by ≤ grid step × 2 |
| **Synthetic Layer B** | Layer A + aggregation at the real 16-region Voronoi group sizes, substation-level demand-space RMSE | Qualitative check of boundary shape only (monotonicity, offset recorded), **no quantitative assertion** |
| **Empirical anchor points** | 240 (baseline, region, signal) combinations from `exp_r25`, positioned by substation-level (alignment, σ_c/σ_r) | Confusion matrix + three propositions, verdict fields rule-generated |

Three variants for Layer A: `geo_log` = renormalization centered in log space (the theorem's strict domain); `arith_log` =
the actual pipeline's arithmetic mass-conserving renormalization + log-space MSE; `arith_demand` = arithmetic renormalization +
demand-space MSE (the actual evaluation convention; recorded only, not asserted).

In [ ]:
# Load persisted artifacts (read-only)
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUT = Path('..') / 'results' / 'exp_r26'
npz = np.load(OUT / 'phase_diagram_data.npz')
emp = pd.read_csv(OUT / 'empirical_points.csv', dtype={'seed': str, 'fold': str})
with open(OUT / 'proposition_checks.json', encoding='utf-8') as f:
    checks = json.load(f)

ALPHAS = npz['alpha_grid']
SRATIOS = npz['sratio_grid']
THEORY = npz['theory_boundary']

for v, e in checks['layer_a']['variants'].items():
    tail = f"assertion {'passed' if e.get('pass') else 'failed'}" if e['asserted'] else 'recorded (not asserted)'
    print(f"Layer A {v:13s}: max boundary deviation {e['max_abs_dev_vs_theory']:.4f} "
          f"(tolerance {e['tolerance']}) -- {tail}")
print(f"Empirical points: {checks['empirical']['n_points']}, "
      f"confusion-matrix accuracy {checks['empirical']['confusion_matrix']['accuracy']:.3f}")

## 1. Synthetic Layer A: Phase Diagram in the Theorem's Strict Domain (Quantitative Assertion Region)

Left panel: the help/hurt phase diagram under the `geo_log` convention (color = sign of the mean ΔMSE), with the black solid line =
the theoretical boundary α\* = σ_ratio/2 and white dots = the empirical boundary (zero-crossing interpolation of ΔMSE).
Right panel: empirical boundary curves for all three conventions vs. the theoretical line — `geo_log` tracks it almost exactly (max deviation ≈ 0.006);
`arith_log` shows a small systematic offset (≈ 0.07, arising from the Jensen gap between the arithmetic renormalization constant log Σw·e^c and
geometric centering); both are within tolerance (2 × 0.05 = 0.1). `arith_demand` retains the boundary shape but shifts upward at high σ_ratio
(≈ 0.19, recorded only); its aggregated counterpart is covered by Layer B.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

# Left: geo_log phase diagram
ax = axes[0]
delta = npz['layerA_delta_geo_log']
lim = np.abs(delta).max()
pc = ax.pcolormesh(SRATIOS, ALPHAS, delta.T, cmap='RdBu_r',
                   vmin=-lim, vmax=lim, shading='nearest')
fig.colorbar(pc, ax=ax, label='Mean ΔMSE (log space)')
ax.plot(SRATIOS, THEORY, 'k-', lw=2, label='Theoretical line α* = σ_ratio/2')
ax.plot(SRATIOS, npz['layerA_boundary_geo_log'], 'o', color='white',
        mec='black', ms=6, label='Empirical boundary (geo_log)')
ax.set_xlabel('σ_c / σ_r'); ax.set_ylabel('α = corr(r, c)')
ax.set_title("Layer A Phase Diagram (geo_log, theorem's strict domain)\nBlue = help (correction improves), red = hurt")
ax.legend(loc='upper left', fontsize=9)

# Right: empirical boundaries (all three conventions) vs. theoretical line
ax = axes[1]
ax.plot(SRATIOS, THEORY, 'k-', lw=2.5, label='Theoretical line α* = σ_ratio/2')
for key, style, lbl in [('layerA_boundary_geo_log', 'o-', 'geo_log (assertion passed)'),
                        ('layerA_boundary_arith_log', 's--', 'arith_log (assertion passed)'),
                        ('layerA_boundary_arith_demand', '^:', 'arith_demand (recorded)')]:
    ax.plot(SRATIOS, npz[key], style, ms=5, alpha=0.85, label=lbl)
tol = checks['layer_a']['variants']['geo_log']['tolerance']
ax.fill_between(SRATIOS, THEORY - tol, THEORY + tol, color='gray', alpha=0.15,
                label=f'Tolerance band ±{tol} (grid step × 2)')
ax.set_xlabel('σ_c / σ_r'); ax.set_ylabel('help/hurt boundary α')
ax.set_title('Layer A Empirical Boundaries (All Three Conventions) vs. the Theoretical Line')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 2. Synthetic Layer B: Adding Voronoi-Style Aggregation (Qualitative)

Agents are grouped and summed at the **real 16-region Voronoi group sizes** (roughly 50k agents → 50–219 groups per region),
and help/hurt is judged by RMSE in substation-level demand space — matching the actual evaluation convention.
Qualitative conclusion: the help/hurt boundary shape is preserved (monotonically non-decreasing, no reversals), with a modest overall
upward shift relative to the theoretical line (the combined effect of aggregation and the demand-space metric); the offset is recorded in
`proposition_checks.json → layer_b.qualitative`, **with no quantitative assertion made**.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2))
fi = npz['layerB_frac_improved']
pc = ax.pcolormesh(SRATIOS, ALPHAS, fi.T, cmap='RdBu', vmin=0, vmax=1,
                   shading='nearest')
fig.colorbar(pc, ax=ax, label='Fraction improved (32 replicates)')
ax.plot(SRATIOS, THEORY, 'k-', lw=2, label='Theoretical line α* = σ_ratio/2')
ax.plot(SRATIOS, npz['layerB_boundary_frac'], 'o', color='white', mec='black',
        ms=6, label='Layer B empirical boundary (fraction improved crosses 0.5)')
ax.set_xlabel('σ_c / σ_r'); ax.set_ylabel('α = corr(r, c)')
ax.set_title('Layer B Phase Diagram (aggregation at real group sizes, substation-level demand-space RMSE)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

q = checks['layer_b']['qualitative']
print(f"Rows with an identifiable boundary: {q['n_rows_with_boundary']}/{q['n_rows_total']}")
print(f"Monotonically non-decreasing (tolerance: 1 grid step): {q['monotonic_nondecreasing_within_one_step']}"
      f" (max reversal {q['max_decrease']:.4f})")
print(f"Offset relative to the theoretical line: median {q['offset_vs_theory_median']:+.4f}, "
      f"max |offset| {q['offset_vs_theory_max_abs']:.4f}")
print(f"Drift from generation coordinates to aggregated measurement coordinates: mean |Δα| = {q['coord_drift_alpha_mean_abs']:.4f}, "
      f"mean |Δσ_ratio| = {q['coord_drift_sratio_mean_abs']:.4f}")

## 3. Overlaying the Empirical Points (240 Combinations → Layer B Coordinate System)

Coordinates = the substation-level measurements from the residual-alignment analysis, (σ_c/σ_r, corr(log F, log ρ)) — using the same
protocol as Layer B's post-aggregation measurement coordinates (ε = regional total demand × 1e-6), so they can be overlaid directly. Small gray dots = Layer B
synthetic replicates (positioned at post-aggregation measurement coordinates; dark gray = hurt, light gray = help).
Filled markers = actual improvement (ΔRMSE < 0), open markers = actual degradation; above the black line = the region where the theory predicts help.

Expected picture: static arms (blue/green) all fall well above the theoretical line and are filled; GNN combination arms (red) crowd
near the theoretical line, with the NP signal (largest σ_ratio) falling below the line and open in large numbers —
"the same set of correction factors meets different fates depending on the mechanism's conditions."

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))

# Layer B replicate background (post-aggregation measurement coordinates)
bh = npz['layerB_points_help'].astype(bool)
ax.scatter(npz['layerB_points_sratio_meas'][bh], npz['layerB_points_alpha_meas'][bh],
           s=4, c='#B8D8C8', alpha=0.35, label='Layer B synthetic (help)', zorder=1)
ax.scatter(npz['layerB_points_sratio_meas'][~bh], npz['layerB_points_alpha_meas'][~bh],
           s=4, c='#C9C9C9', alpha=0.35, label='Layer B synthetic (hurt)', zorder=1)

# Theoretical line
xs = np.linspace(0, 1.6, 100)
ax.plot(xs, xs / 2, 'k-', lw=2, label='Theoretical line α* = σ_ratio/2', zorder=3)

# 240 empirical points
colors = {'uniform': '#4C72B0', 'gpm': '#55A868', 'gnn': '#C44E52'}
markers = {'uniform': 'o', 'gpm': 's', 'gnn': '^'}
for bt in ('uniform', 'gpm', 'gnn'):
    sub = emp[emp.base_type == bt]
    imp = sub['actual_help']
    ax.scatter(sub.loc[imp, 'sigma_ratio'], sub.loc[imp, 'pearson_log'],
               c=colors[bt], marker=markers[bt], s=42, ec='black', lw=0.4,
               label=f'{bt} (actual improvement)', zorder=4)
    ax.scatter(sub.loc[~imp, 'sigma_ratio'], sub.loc[~imp, 'pearson_log'],
               facecolors='none', edgecolors=colors[bt], marker=markers[bt],
               s=42, lw=1.3, label=f'{bt} (actual degradation)', zorder=4)

ax.set_xlabel('σ_c / σ_r (substation-level measurement)')
ax.set_ylabel('Alignment α = corr(log F, log ρ)')
ax.set_title('Overlay of Empirical Points: Above the Theoretical Line = Predicted Help Region')
ax.set_xlim(0, 1.6); ax.set_ylim(-0.35, 1.0)
ax.legend(fontsize=8, loc='lower right', ncol=2)
plt.tight_layout(); plt.show()

## 4. Confusion Matrix and the Three Propositions (Verdict Fields Are Rule-Generated)

- **Prediction rule**: predicted_help ⟺ pearson_log > sigma_ratio/2;
  **Actual rule**: actual_help ⟺ ΔRMSE < 0.
- **Verdict rule**: value ≥ 0.9 → supported; ≥ 0.5 → partially_supported;
  otherwise not_supported (the definition of `value` is given in each proposition's field).

In [ ]:
cm = checks['empirical']['confusion_matrix']
mat = np.array([[cm['pred_help_actual_help'], cm['pred_help_actual_hurt']],
                [cm['pred_hurt_actual_help'], cm['pred_hurt_actual_hurt']]])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
im = ax.imshow(mat, cmap='Blues')
for (i, j), v in np.ndenumerate(mat):
    ax.text(j, i, str(v), ha='center', va='center', fontsize=18,
            color='white' if v > mat.max() / 2 else 'black')
ax.set_xticks([0, 1], ['Actual improved', 'Actual degraded'])
ax.set_yticks([0, 1], ['Predicted help', 'Predicted hurt'])
ax.set_title(f"Confusion Matrix (total {cm['total']}, accuracy {cm['accuracy']:.3f})")

ax = axes[1]
rows = []
for bt, c in checks['empirical']['confusion_by_base_type'].items():
    rows.append([bt, c['pred_help_actual_help'], c['pred_help_actual_hurt'],
                 c['pred_hurt_actual_help'], c['pred_hurt_actual_hurt'],
                 f"{c['accuracy']:.3f}"])
tbl = ax.table(cellText=rows, colLabels=['Base', 'H/H', 'H/hurt', 'hurt/H',
                                         'hurt/hurt', 'Accuracy'],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.6)
ax.axis('off'); ax.set_title('Breakdown by Base Type')
plt.tight_layout(); plt.show()

print('── Verdicts for the Three Propositions ──')
for name, p in checks['empirical']['propositions'].items():
    print(f"\n{name}: {p['description']}")
    for k, v in p.items():
        if k in ('description', 'by_base_type'):
            continue
        print(f"  {k} = {v}")
    if 'by_base_type' in p:
        for bt, d in p['by_base_type'].items():
            print(f"  {bt}: {d}")

## 5. Summary (Numbers Follow the Values Persisted in `proposition_checks.json`)

1. **Layer A (quantitative)**: the theoretical line α\* = σ_c/(2σ_r), within its strict domain (log space, c centered),
   is reproduced precisely by the empirical boundary (geo_log max deviation ≈ 0.006 ≪ tolerance 0.1); switching to the actual pipeline's
   arithmetic renormalization stays within tolerance as well (arith_log ≈ 0.07). Under the demand-space convention (arith_demand), the
   boundary shape is preserved but shifts upward at high σ_ratio — meaning multiplicative correction in demand space is **more prone to being harmful**
   than the log-space theory predicts, which reinforces rather than weakens the intent of the conditional proposition.
2. **Layer B (qualitative)**: after aggregation at the real group sizes, the boundary is monotonically non-decreasing and its shape is preserved,
   with a modest overall upward shift relative to the theoretical line (median offset given in the JSON) — the theoretical line, under the
   aggregated convention, is a reasonable approximate **necessary** direction for help, and is a valid qualitative reference.
3. **Empirical**: the 240-point confusion matrix has an accuracy of 0.833; the static arms fall in the help region in 96/96 cases and all
   show actual improvement (Proposition 1 supported); the GNN×NP arm falls in the hurt region 58% of the time and shows actual degradation 77%
   of the time (Proposition 2 partially supported — the points crowd near the theoretical line, consistent with the mechanism narrative that
   "a small σ_r under the GNN baseline raises the effective threshold"); the combined-signal variance is superadditive in 80/80 cases
   (Proposition 3 supported, with the median σ_c(NP)² reaching 1.63× σ_c(N)²+σ_c(P)²) — the positive correlation between the N and P factors
   makes σ_c grow superlinearly for the combined arm, explaining why the combined arm is the first to cross the hurt boundary on the GNN baseline.